# Day 17 — 탐색적 분석: 분포, 정규화, PCA

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-3/day-17.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-3/day-17.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — housekeeping으로 나누면 투입량 차이가 사라진다

In [ ]:
import numpy as np
import pandas as pd

raw = pd.DataFrame(
    {"HK1": [100, 200], "HK2": [400, 800], "GeneX": [50, 100], "GeneY": [30, 120]},
    index=["sampleA", "sampleB"],            # sampleB에는 RNA가 2배 들어갔다고 가정
)
hk_geo = np.exp(np.log(raw[["HK1", "HK2"]]).mean(axis=1))    # 기하평균
size_factor = hk_geo / hk_geo.mean()
norm = raw[["GeneX", "GeneY"]].div(size_factor, axis=0)

print(size_factor.round(2).to_string())
print(norm.round(1).to_string())

### 예시 2 — log2를 취하면 "두 배"가 같은 간격이 된다

In [ ]:
counts = np.array([0, 9, 19, 39, 1023, 2047])
print(np.log2(counts + 1))

### 예시 3 — 아주 작은 PCA

In [ ]:
from sklearn.decomposition import PCA

toy = pd.DataFrame(
    {"g1": [5.0, 5.2, 4.9, 7.1, 7.0, 6.8],
     "g2": [3.0, 3.1, 2.9, 1.0, 1.2, 0.9],
     "g3": [6.0, 6.1, 5.9, 6.0, 6.1, 6.0]},
    index=["A1", "A2", "A3", "B1", "B2", "B3"],
)
pca = PCA(n_components=2)
scores = pd.DataFrame(pca.fit_transform(toy), index=toy.index, columns=["PC1", "PC2"])
print(scores.round(2).to_string())
print("설명된 분산 비율:", pca.explained_variance_ratio_.round(3))

## 직접 해보기

1. 설정 셀과 어제 저장한 표를 불러옵니다.

In [ ]:
# 이 셀은 Day 15–16의 산출물 파일이 필요합니다. Colab은 노트북마다 런타임이 따로라서,
# 그 파일을 이 런타임에 업로드하거나 Google Drive에 두고 경로를 맞추세요.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

MODE = "practice"          # 연습이 끝나면 "real"로 바꾸세요
RAW_DIR = Path("../data/osdr_536_practice" if MODE == "practice" else "../data/osdr_536")
OUT_DIR = RAW_DIR / "processed"
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

counts = pd.read_csv(OUT_DIR / "counts_raw.csv", index_col=0)
probes = pd.read_csv(OUT_DIR / "probes.csv")
samples = pd.read_csv(OUT_DIR / "samples.csv").set_index("sample_id").loc[counts.index]

def names(code_class):
    return probes.loc[probes["CodeClass"] == code_class, "Name"].tolist()

endo, hk, neg, pos = (counts[names(c)] for c in ["Endogenous", "Housekeeping", "Negative", "Positive"])
colors = samples["group"].map({"FLT": "tab:red", "GC": "tab:blue"})
print(endo.shape, hk.shape, neg.shape, pos.shape)

2. 샘플별 QC 표를 만듭니다. 배경 기준은 "Negative 대조의 평균 + 2 × 표준편차"로 잡습니다.

In [ ]:
hk_use = hk.columns[hk.min() >= 20]                 # 너무 낮은 housekeeping은 기준자로 쓰지 않음
hk_geo = np.exp(np.log(hk[hk_use]).mean(axis=1))
background = neg.mean(axis=1) + 2 * neg.std(axis=1)

sample_qc = pd.DataFrame({
    "source": samples["source"],
    "group": samples["group"],
    "endo_total": endo.sum(axis=1),
    "hk_geomean": hk_geo.round(1),
    "size_factor": (hk_geo / hk_geo.mean()).round(2),
    "background": background.round(1),
    "pos_max": pos.max(axis=1),
    "pct_detected": (endo.gt(background, axis=0).mean(axis=1) * 100).round(1),
})
print("사용한 housekeeping:", len(hk_use), "/", hk.shape[1])
print(sample_qc.to_string())

3. 정규화하고 log2로 바꿉니다. 배경보다 높게 잡힌 샘플이 절반 미만인 유전자는 "측정이 불안정한 유전자"로 표시해 둡니다.

In [ ]:
norm = endo.div(sample_qc["size_factor"], axis=0)
log2_raw = np.log2(endo + 1)
log2_norm = np.log2(norm + 1)

gene_qc = pd.DataFrame({
    "detected_frac": endo.gt(background, axis=0).mean(),
    "mean_log2": log2_norm.mean(),
})
gene_qc["keep"] = gene_qc["detected_frac"] >= 0.5
print("전체 유전자:", len(gene_qc), "| 분석에 남길 유전자:", int(gene_qc["keep"].sum()))

4. 샘플별 boxplot을 정규화 전후로 나란히 그립니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, data, title in [(axes[0], log2_raw, "Before normalization"),
                        (axes[1], log2_norm, "After housekeeping normalization")]:
    bp = ax.boxplot(data.T.values, patch_artist=True)
    for box, c in zip(bp["boxes"], colors):
        box.set_facecolor(c)
        box.set_alpha(0.6)
    ax.set_xticks(range(1, len(data) + 1), samples["source"], rotation=90)
    ax.set_title(title)
axes[0].set_ylabel("log2(count + 1)")
fig.suptitle(f"Per-sample distribution (red = FLT, blue = GC) [data: {MODE}]")
fig.tight_layout()
fig.savefig(FIG_DIR / f"osdr_{MODE}_boxplot.png", dpi=150)

5. density plot을 그립니다. 선 하나가 샘플 하나입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for sid in log2_norm.index:
    log2_norm.loc[sid].plot.density(ax=ax, color=colors[sid], alpha=0.7, linewidth=1)
ax.set_xlabel("log2(normalized count + 1)")
ax.set_title(f"Density per sample (red = FLT, blue = GC) [data: {MODE}]")
fig.tight_layout()
fig.savefig(FIG_DIR / f"osdr_{MODE}_density.png", dpi=150)

6. 남긴 유전자로 PCA를 그리고, 좌표를 표로도 봅니다.

In [ ]:
X = log2_norm.loc[:, gene_qc["keep"]]
pca = PCA(n_components=2)
pcs = pd.DataFrame(pca.fit_transform(X), index=X.index, columns=["PC1", "PC2"])
var = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(pcs["PC1"], pcs["PC2"], c=colors, s=60)
for sid, row in pcs.iterrows():
    ax.annotate(samples.loc[sid, "source"], (row["PC1"], row["PC2"]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel(f"PC1 ({var[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
ax.set_title(f"PCA of normalized log2 counts (red = FLT, blue = GC) [data: {MODE}]")
fig.tight_layout()
fig.savefig(FIG_DIR / f"osdr_{MODE}_pca.png", dpi=150)

center = pcs.groupby(samples["group"]).transform("mean")       # 자기 군의 중심
dist = np.sqrt(((pcs - center) ** 2).sum(axis=1))
sample_qc["dist_ratio"] = (dist / dist.median()).round(2)
print(f"PC1 {var[0]:.1f}%, PC2 {var[1]:.1f}%")
print(sample_qc[["source", "group", "size_factor", "pct_detected", "dist_ratio"]].to_string())

7. 결과를 저장합니다.

In [ ]:
log2_norm.to_csv(OUT_DIR / "log2_norm.csv")
gene_qc.to_csv(OUT_DIR / "gene_qc.csv")
sample_qc.to_csv(OUT_DIR / "sample_qc.csv")
print("저장 완료:", sorted(p.name for p in OUT_DIR.glob("*.csv")))